# 34. Neural Networks: Convolutional Neural Networks (CNNs)

## Algorithm Category
**Type**: Neural Networks - Deep Learning  
**Complexity**: High  
**Use Case**: Image classification, object detection, computer vision tasks

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand convolutional layers and their operations
- Implement CNNs using PyTorch
- Understand pooling, padding, and stride
- Visualize learned filters and feature maps
- Apply CNNs to image classification
- Understand transfer learning with pre-trained models

## Historical Context

CNNs were inspired by biological vision:
- LeCun, Y., et al. (1998): "Gradient-based learning applied to document recognition"
- LeNet-5 (1998): First successful CNN for digit recognition
- AlexNet (2012): Breakthrough in ImageNet competition
- Foundation for modern computer vision

**Key Papers/References:**
- LeCun, Y., et al. (1998). "Gradient-based learning applied to document recognition"
- Krizhevsky, A., et al. (2012). "ImageNet classification with deep convolutional neural networks"

## When to Use CNNs

CNNs are appropriate when:
- Working with image data
- Need spatial feature extraction
- Object detection and recognition
- Computer vision tasks
- When data has grid-like structure (images, time series)
- Transfer learning from pre-trained models

## Theory & Mechanics

### Mathematical Foundation

**Convolution Operation:**
$$(f * g)(x, y) = \sum_{i} \sum_{j} f(i, j) \cdot g(x-i, y-j)$$

**Convolutional Layer:**
$$h_{i,j} = \sum_{m} \sum_{n} x_{i+m, j+n} \cdot w_{m,n} + b$$

**Max Pooling:**
$$h_{i,j} = \max_{m,n \in \text{pool}} x_{i+m, j+n}$$

### Key Components

1. **Convolutional Layers**
   - Apply filters (kernels) to input
   - Detect local patterns (edges, textures)
   - Share weights across spatial locations
   - Reduce parameters compared to fully connected

2. **Pooling Layers**
   - Downsample feature maps
   - Reduce spatial dimensions
   - Max pooling: take maximum value
   - Average pooling: take average value

3. **Activation Functions**
   - ReLU: Introduces non-linearity
   - Applied after convolution

4. **Fully Connected Layers**
   - Final classification layers
   - Flatten feature maps

### How It Works

1. **Convolution**: Apply filters to detect features
2. **Activation**: Apply ReLU for non-linearity
3. **Pooling**: Downsample to reduce size
4. **Repeat**: Stack multiple conv layers
5. **Flatten**: Convert to 1D for classification
6. **Dense**: Final classification layers

### Key Hyperparameters

- **filters**: Number of convolutional filters
- **kernel_size**: Size of convolution kernel
- **stride**: Step size for convolution
- **padding**: Border handling ('same', 'valid')
- **pool_size**: Size of pooling window
- **dropout**: Regularization rate

### Advantages

- Translation invariant
- Parameter sharing (efficient)
- Hierarchical feature learning
- Excellent for images
- Can use transfer learning

### Limitations

- Requires large datasets
- Computationally expensive
- Black box (hard to interpret)
- Sensitive to input size
- Requires GPU for training


## Implementation

Let's implement CNNs using PyTorch for image classification.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# PyTorch: Deep learning framework
import torch  # PyTorch core library (tensors, automatic differentiation)
import torch.nn as nn  # Neural network layers and modules
import torch.optim as optim  # Optimization algorithms (Adam, SGD, etc.)
from torch.utils.data import DataLoader, TensorDataset  # Data loading utilities

# Scikit-learn: Machine learning utilities
from sklearn.datasets import fetch_openml  # Download datasets from OpenML
from sklearn.model_selection import train_test_split  # Split data into train/test sets
from sklearn.preprocessing import StandardScaler  # Feature scaling (not used here but good practice)
from sklearn.metrics import accuracy_score, classification_report  # Evaluation metrics

# ============================================
# GPU DETECTION: Using CUDA for Faster Training
# ============================================

# Check if CUDA (GPU) is available
# CNNs are computationally intensive - GPU acceleration makes training much faster
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# torch.cuda.is_available(): Returns True if GPU is available
# If GPU available: device = 'cuda' (use GPU)
# If no GPU: device = 'cpu' (use CPU - slower but still works)
print(f"Using device: {device}")  # Display which device we're using

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# DEFINING CNN ARCHITECTURE: Building the Model
# ============================================

# SimpleCNN: A basic convolutional neural network for image classification
# This architecture follows the classic CNN pattern: Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> FC -> FC
class SimpleCNN(nn.Module):
    """
    Simple CNN for image classification.
    
    Architecture:
    - Conv Block 1: 1 channel -> 32 channels (feature extraction)
    - Conv Block 2: 32 channels -> 64 channels (deeper features)
    - Fully Connected: 64*7*7 -> 128 -> num_classes (classification)
    """
    
    def __init__(self, num_classes=10):
        """
        Initialize the CNN model.
        
        Args:
            num_classes: Number of output classes (default: 10 for MNIST digits)
        """
        super(SimpleCNN, self).__init__()
        # super() calls the parent class (nn.Module) constructor
        # This is required for PyTorch modules
        
        # ============================================
        # CONVOLUTIONAL LAYERS: Feature Extraction
        # ============================================
        
        # Conv Layer 1: First feature extraction layer
        # nn.Conv2d(in_channels, out_channels, kernel_size, padding)
        # - in_channels=1: Input has 1 channel (grayscale images)
        # - out_channels=32: Output has 32 feature maps (32 different filters)
        # - kernel_size=3: 3×3 convolution filter (detects local patterns)
        # - padding=1: Add 1 pixel border (keeps output size same as input)
        #   Without padding, 28×28 -> 26×26. With padding=1, 28×28 -> 28×28
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        # This layer learns 32 different filters (edge detectors, texture detectors, etc.)
        
        # Conv Layer 2: Deeper feature extraction
        # - in_channels=32: Input from previous layer (32 feature maps)
        # - out_channels=64: Output has 64 feature maps (more complex patterns)
        # - kernel_size=3: 3×3 filter (same as before)
        # - padding=1: Keep spatial dimensions
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # This layer learns more complex patterns (combinations of simple features)
        
        # ============================================
        # POOLING LAYER: Downsampling
        # ============================================
        
        # Max Pooling: Reduces spatial dimensions by taking maximum value in each window
        # nn.MaxPool2d(kernel_size, stride)
        # - kernel_size=2: 2×2 pooling window
        # - stride=2: Move window by 2 pixels (no overlap)
        # Effect: 28×28 -> 14×14 (halves both dimensions)
        self.pool = nn.MaxPool2d(2, 2)
        # Max pooling preserves strongest features and reduces computation
        
        # ============================================
        # FULLY CONNECTED LAYERS: Classification
        # ============================================
        
        # FC Layer 1: First dense layer
        # nn.Linear(in_features, out_features)
        # - in_features=64*7*7: After 2 pooling operations, 28×28 -> 7×7
        #   (28 -> 14 -> 7 after two 2×2 pools with stride 2)
        # - out_features=128: Hidden layer with 128 neurons
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        # This layer combines all spatial features into a single vector
        
        # FC Layer 2: Output layer (classification)
        # - in_features=128: From previous layer
        # - out_features=num_classes: One output per class (e.g., 10 for digits 0-9)
        self.fc2 = nn.Linear(128, num_classes)
        # This layer produces class scores (logits) for each class
        
        # ============================================
        # ACTIVATION AND REGULARIZATION
        # ============================================
        
        # ReLU: Rectified Linear Unit activation function
        # ReLU(x) = max(0, x) - introduces non-linearity
        # Applied after convolutional and fully connected layers
        self.relu = nn.ReLU()
        # Non-linearity is essential - without it, the network would be just a linear transformation
        
        # Dropout: Regularization technique to prevent overfitting
        # Randomly sets 50% of neurons to zero during training
        # Prevents network from memorizing training data
        self.dropout = nn.Dropout(0.5)
        # 0.5 = 50% dropout rate (half the neurons are randomly disabled)
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, 1, 28, 28)
        
        Returns:
            Output tensor of shape (batch_size, num_classes)
        """
        # ============================================
        # CONV BLOCK 1: First Feature Extraction
        # ============================================
        
        # Apply first convolutional layer
        # Input: (batch, 1, 28, 28) -> Output: (batch, 32, 28, 28)
        x = self.conv1(x)
        # Convolution detects local patterns (edges, corners, textures)
        
        # Apply ReLU activation
        # ReLU introduces non-linearity (allows network to learn complex patterns)
        x = self.relu(x)
        # Negative values become 0, positive values stay the same
        
        # Apply max pooling
        # Input: (batch, 32, 28, 28) -> Output: (batch, 32, 14, 14)
        x = self.pool(x)
        # Reduces spatial dimensions (28×28 -> 14×14)
        # Preserves strongest features in each 2×2 region
        
        # ============================================
        # CONV BLOCK 2: Deeper Feature Extraction
        # ============================================
        
        # Apply second convolutional layer
        # Input: (batch, 32, 14, 14) -> Output: (batch, 64, 14, 14)
        x = self.conv2(x)
        # Learns more complex patterns (combinations of simple features)
        
        # Apply ReLU activation
        x = self.relu(x)
        
        # Apply max pooling
        # Input: (batch, 64, 14, 14) -> Output: (batch, 64, 7, 7)
        x = self.pool(x)
        # Further reduces spatial dimensions (14×14 -> 7×7)
        
        # ============================================
        # FLATTEN: Convert 2D Feature Maps to 1D Vector
        # ============================================
        
        # Flatten feature maps into a 1D vector
        # x.view(-1, 64 * 7 * 7): Reshape to (batch_size, 64*7*7)
        # -1: Automatically calculate batch size
        # 64*7*7 = 3136: Total number of features (64 channels × 7×7 spatial)
        x = x.view(-1, 64 * 7 * 7)
        # Converts (batch, 64, 7, 7) -> (batch, 3136)
        # This prepares data for fully connected layers (which need 1D input)
        
        # ============================================
        # FULLY CONNECTED LAYERS: Classification
        # ============================================
        
        # First fully connected layer
        # Input: (batch, 3136) -> Output: (batch, 128)
        x = self.fc1(x)
        # Combines all spatial features into a single representation
        
        # Apply ReLU activation
        x = self.relu(x)
        
        # Apply dropout (only during training, disabled during evaluation)
        # Randomly sets 50% of neurons to zero (prevents overfitting)
        x = self.dropout(x)
        # During training: some neurons are randomly disabled
        # During evaluation: all neurons are active (dropout is automatically disabled)
        
        # Output layer (classification)
        # Input: (batch, 128) -> Output: (batch, num_classes)
        x = self.fc2(x)
        # Produces class scores (logits) for each class
        # Higher score = more likely that class
        
        return x  # Return class scores

print("CNN model defined!")  # Confirm model is ready


In [ ]:
# ============================================
# LOADING DATASET: MNIST Handwritten Digits
# ============================================

# MNIST is a classic dataset of 70,000 handwritten digit images (0-9)
# Each image is 28×28 pixels (grayscale)
# This is the "Hello World" of computer vision - perfect for learning CNNs

# Try to load MNIST from OpenML (online repository)
try:
    # fetch_openml() downloads dataset from OpenML
    # 'mnist_784': MNIST dataset (784 = 28×28 pixels flattened)
    # version=1: Specific dataset version
    # as_frame=False: Return as NumPy array (not pandas DataFrame)
    # parser='auto': Automatic parser selection
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X, y = mnist.data, mnist.target.astype(int)
    # X: Feature data (images flattened to 784 pixels)
    # y: Labels (digit class: 0-9)
    # .astype(int): Convert labels from string to integer
    print(f"MNIST dataset loaded: {X.shape}")  # Should be (70000, 784)
except Exception as e:
    # If download fails (no internet, etc.), create synthetic data
    print(f"Error loading MNIST: {e}")
    print("Creating synthetic image data for demonstration...")
    # Create synthetic 28×28 images (random data for demonstration)
    X = np.random.rand(1000, 784).astype(np.float32)  # 1000 random images
    y = np.random.randint(0, 10, 1000)  # Random labels (0-9)

# ============================================
# DATA SUBSET: Using Smaller Sample for Faster Training
# ============================================

# Use subset for faster training (CNNs can be slow on full dataset)
# In practice, you'd use the full dataset for better performance
n_samples = min(5000, len(X))  # Use 5000 samples or all if less
X = X[:n_samples]  # Take first n_samples
y = y[:n_samples]  # Take corresponding labels

# ============================================
# RESHAPING DATA: Converting Flattened Images to 2D
# ============================================

# Reshape from flattened (784,) to 2D images (28, 28)
# Original: (n_samples, 784) - each image is a flat array
# Reshaped: (n_samples, 28, 28) - each image is a 2D array
X_images = X.reshape(-1, 28, 28)
# -1: Automatically calculate number of samples
# 28, 28: Height and width of each image

# ============================================
# NORMALIZATION: Scaling Pixel Values to [0, 1]
# ============================================

# Normalize pixel values from [0, 255] to [0, 1]
# This helps neural networks train faster and more stably
# Neural networks work better with normalized inputs
X_images = X_images / 255.0
# Original: pixel values 0-255 (8-bit grayscale)
# Normalized: pixel values 0.0-1.0 (float)

# ============================================
# TRAIN/TEST SPLIT: Separating Training and Testing Data
# ============================================

# Split data into training (80%) and testing (20%) sets
# train_test_split() randomly divides data
X_train, X_test, y_train, y_test = train_test_split(
    X_images, y, test_size=0.2, random_state=42
)
# X_images: Feature data (images)
# y: Labels (digit classes)
# test_size=0.2: 20% for testing, 80% for training
# random_state=42: Seed for reproducibility (same split every time)

# ============================================
# CONVERTING TO PYTORCH TENSORS: Preparing for Neural Network
# ============================================

# PyTorch uses tensors (like NumPy arrays but with GPU support)
# Convert NumPy arrays to PyTorch tensors

# Training features: Convert to FloatTensor (float32)
X_train_tensor = torch.FloatTensor(X_train).unsqueeze(1)
# torch.FloatTensor(): Convert NumPy array to PyTorch float tensor
# .unsqueeze(1): Add channel dimension
# Original shape: (n_train, 28, 28)
# After unsqueeze: (n_train, 1, 28, 28) - (batch, channels, height, width)
# CNNs expect 4D tensors: (batch, channels, height, width)

# Test features: Same conversion
X_test_tensor = torch.FloatTensor(X_test).unsqueeze(1)
# Shape: (n_test, 1, 28, 28)

# Training labels: Convert to LongTensor (int64, required for classification)
y_train_tensor = torch.LongTensor(y_train)
# Shape: (n_train,) - one label per image

# Test labels: Same conversion
y_test_tensor = torch.LongTensor(y_test)
# Shape: (n_test,)

# ============================================
# DISPLAYING DATA INFORMATION
# ============================================

print(f"Training set: {X_train_tensor.shape}")  # Should be (n_train, 1, 28, 28)
print(f"Test set: {X_test_tensor.shape}")  # Should be (n_test, 1, 28, 28)
print(f"Number of classes: {len(np.unique(y))}")  # Should be 10 (digits 0-9)


In [ ]:
# ============================================
# CREATING DATA LOADERS: Batching for Efficient Training
# ============================================

# Data loaders organize data into batches for efficient training
# Instead of processing one image at a time, we process batches (faster!)

# TensorDataset: Combines features and labels into a dataset
# This is a PyTorch utility that pairs inputs with targets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# Pairs each image (X_train_tensor[i]) with its label (y_train_tensor[i])

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
# Same for test data

# DataLoader: Creates batches and handles shuffling
# train_loader: For training (shuffled batches)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# batch_size=32: Process 32 images at once (faster than one-by-one)
# shuffle=True: Randomly shuffle data each epoch (prevents overfitting to order)

# test_loader: For testing (no shuffling needed)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
# batch_size=32: Same batch size for consistency
# shuffle=False: Don't shuffle test data (order doesn't matter for evaluation)

# ============================================
# INITIALIZING MODEL: Creating and Moving to Device
# ============================================

# Create CNN model instance
# num_classes: Number of output classes (10 for digits 0-9)
model = SimpleCNN(num_classes=len(np.unique(y))).to(device)
# SimpleCNN(): Create model instance
# .to(device): Move model to GPU (if available) or CPU
# This is important - model and data must be on same device!

# ============================================
# LOSS FUNCTION: Cross-Entropy for Classification
# ============================================

# CrossEntropyLoss: Standard loss function for multi-class classification
# Combines LogSoftmax and NLLLoss in one efficient function
criterion = nn.CrossEntropyLoss()
# This loss function:
# 1. Takes model outputs (logits: raw scores for each class)
# 2. Takes true labels (class indices: 0-9)
# 3. Computes how wrong the predictions are
# Lower loss = better predictions

# ============================================
# OPTIMIZER: Adam for Gradient Descent
# ============================================

# Adam optimizer: Adaptive learning rate optimizer
# Better than basic SGD (Stochastic Gradient Descent) for most cases
# Automatically adjusts learning rate for each parameter
optimizer = optim.Adam(model.parameters(), lr=0.001)
# model.parameters(): All trainable weights in the model
# lr=0.001: Learning rate (step size for weight updates)
# Lower lr = slower but more stable learning
# Higher lr = faster but may overshoot optimal weights

# ============================================
# DISPLAYING MODEL INFORMATION
# ============================================

print(f"Model architecture:")
print(model)  # Print model structure (layers and dimensions)

# Count total number of parameters (weights) in the model
total_params = sum(p.numel() for p in model.parameters())
# p.numel(): Number of elements in parameter tensor
# sum(): Total across all parameters
print(f"\nTotal parameters: {total_params:,}")
# More parameters = more capacity to learn, but also more risk of overfitting
# This CNN has relatively few parameters compared to fully connected networks!


## Training

Let's train the CNN model.

### Understanding the Training Process

The training process involves:
1. **Forward Pass**: Images flow through the network to produce predictions
2. **Loss Calculation**: Compare predictions with true labels
3. **Backward Pass**: Compute gradients (how to adjust weights)
4. **Weight Update**: Adjust weights to reduce loss

This process repeats for many epochs until the model learns to recognize digits.


In [ ]:
# ============================================
# TRAINING LOOP: Learning from Data
# ============================================

# Training parameters
num_epochs = 5  # Number of times to go through entire training dataset
# One epoch = one complete pass through all training data
# More epochs = more learning, but risk of overfitting

# Lists to track training progress
train_losses = []  # Store loss for each epoch
train_accuracies = []  # Store accuracy for each epoch

# ============================================
# EPOCH LOOP: Training Over Multiple Epochs
# ============================================

for epoch in range(num_epochs):
    # Set model to training mode
    # This enables dropout and batch normalization training behavior
    model.train()
    # In training mode:
    # - Dropout is active (randomly disables neurons)
    # - Batch normalization uses batch statistics
    
    # Reset statistics for this epoch
    running_loss = 0.0  # Accumulate loss across batches
    correct = 0  # Count correct predictions
    total = 0  # Count total predictions
    
    # ============================================
    # BATCH LOOP: Processing Data in Batches
    # ============================================
    
    for batch_idx, (data, target) in enumerate(train_loader):
        # data: Batch of images (batch_size, 1, 28, 28)
        # target: Batch of labels (batch_size,)
        
        # Move data to GPU (if available) for faster computation
        data, target = data.to(device), target.to(device)
        # .to(device): Transfer tensors to GPU or CPU
        # Model and data must be on same device!
        
        # ============================================
        # FORWARD PASS: Making Predictions
        # ============================================
        
        # Zero gradients from previous iteration
        # PyTorch accumulates gradients - must zero them before each backward pass
        optimizer.zero_grad()
        # If we don't zero gradients, they accumulate across iterations (wrong!)
        
        # Forward pass: Run images through the network
        output = model(data)
        # Input: (batch_size, 1, 28, 28) - batch of images
        # Output: (batch_size, 10) - class scores for each image
        # Each row has 10 scores (one per digit class 0-9)
        
        # Calculate loss: How wrong are our predictions?
        loss = criterion(output, target)
        # output: Model predictions (class scores)
        # target: True labels (digit classes: 0-9)
        # loss: Single number measuring prediction error
        # Lower loss = better predictions
        
        # ============================================
        # BACKWARD PASS: Computing Gradients
        # ============================================
        
        # Backward pass: Compute gradients (derivatives)
        # This calculates how to adjust each weight to reduce loss
        loss.backward()
        # Computes gradients for all parameters using chain rule (backpropagation)
        # Gradients tell us: "Which direction should we adjust each weight?"
        
        # Update weights: Move weights in direction that reduces loss
        optimizer.step()
        # Uses computed gradients to update all model parameters
        # Adam optimizer: Adaptive learning rate (adjusts step size per parameter)
        # This is the actual "learning" - weights are updated!
        
        # ============================================
        # STATISTICS: Tracking Performance
        # ============================================
        
        # Accumulate loss for this batch
        running_loss += loss.item()
        # loss.item(): Extract scalar value from tensor (convert to Python float)
        
        # Calculate predictions: Which class has highest score?
        _, predicted = torch.max(output.data, 1)
        # torch.max(output.data, 1): Find class with maximum score for each image
        # Returns: (max_values, max_indices)
        # We only need max_indices (predicted class), so use _ for max_values
        # predicted: (batch_size,) - predicted class for each image
        
        # Count total and correct predictions
        total += target.size(0)  # Number of images in this batch
        correct += (predicted == target).sum().item()
        # (predicted == target): Boolean tensor (True where prediction matches label)
        # .sum(): Count how many are True (correct predictions)
        # .item(): Convert to Python int
    
    # ============================================
    # EPOCH SUMMARY: Average Performance
    # ============================================
    
    # Calculate average loss for this epoch
    epoch_loss = running_loss / len(train_loader)
    # Total loss / number of batches = average loss per batch
    
    # Calculate accuracy for this epoch
    epoch_acc = 100 * correct / total
    # (correct predictions / total predictions) × 100 = accuracy percentage
    
    # Store metrics for plotting
    train_losses.append(epoch_loss)  # Save loss for this epoch
    train_accuracies.append(epoch_acc)  # Save accuracy for this epoch
    
    # Display progress
    print(f"Epoch {epoch+1}/{num_epochs}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%")
    # Shows how well model is learning
    # Loss should decrease, accuracy should increase over epochs

print("\nTraining complete!")  # All epochs finished


## Evaluation

Let's evaluate the model on test data.


In [ ]:
# ============================================
# EVALUATION: Testing Model Performance
# ============================================

# Set model to evaluation mode
# This disables dropout and uses batch normalization statistics from training
model.eval()
# In evaluation mode:
# - Dropout is disabled (all neurons active)
# - Batch normalization uses running statistics (not batch statistics)
# - No gradient computation needed (faster, uses less memory)

# Initialize counters for evaluation
test_correct = 0  # Count correct predictions
test_total = 0  # Count total predictions
all_predictions = []  # Store all predictions (for detailed analysis)
all_targets = []  # Store all true labels (for detailed analysis)

# ============================================
# EVALUATION LOOP: Testing on Unseen Data
# ============================================

# torch.no_grad(): Disable gradient computation (faster, saves memory)
# We don't need gradients during evaluation (not updating weights)
with torch.no_grad():
    # Process test data in batches
    for data, target in test_loader:
        # data: Batch of test images (batch_size, 1, 28, 28)
        # target: Batch of true labels (batch_size,)
        
        # Move data to GPU (if available)
        data, target = data.to(device), target.to(device)
        
        # Forward pass: Get model predictions
        output = model(data)
        # Input: (batch_size, 1, 28, 28) - test images
        # Output: (batch_size, 10) - class scores for each image
        
        # Get predicted class: Which class has highest score?
        _, predicted = torch.max(output.data, 1)
        # torch.max(output.data, 1): Find class with maximum score
        # Returns: (max_values, max_indices)
        # We only need max_indices (predicted class)
        # predicted: (batch_size,) - predicted digit for each image
        
        # Update statistics
        test_total += target.size(0)  # Number of images in this batch
        test_correct += (predicted == target).sum().item()
        # (predicted == target): Boolean tensor (True where correct)
        # .sum(): Count correct predictions
        # .item(): Convert to Python int
        
        # Store predictions and labels for detailed analysis
        all_predictions.extend(predicted.cpu().numpy())
        # .cpu(): Move tensor to CPU (if it was on GPU)
        # .numpy(): Convert to NumPy array
        # .extend(): Add to list
        all_targets.extend(target.cpu().numpy())
        # Store true labels for comparison

# ============================================
# CALCULATING TEST ACCURACY
# ============================================

# Calculate overall test accuracy
test_accuracy = 100 * test_correct / test_total
# (correct predictions / total predictions) × 100 = accuracy percentage
# This tells us: "What percentage of test images were classified correctly?"

print(f"Test Accuracy: {test_accuracy:.2f}%")
# Higher accuracy = better model
# For MNIST, good models achieve >95% accuracy
# Random guessing would be ~10% (1 out of 10 classes)

# ============================================
# DETAILED CLASSIFICATION REPORT
# ============================================

# Classification report: Detailed metrics per class
print("\nClassification Report:")
print(classification_report(all_targets, all_predictions))
# Shows for each digit class (0-9):
# - Precision: Of predicted digit X, how many were actually X?
# - Recall: Of actual digit X, how many did we find?
# - F1-Score: Harmonic mean of precision and recall
# - Support: Number of samples of each class
# This helps identify which digits are harder to classify


## Visualization

Let's visualize training progress and sample predictions.


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'o-')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_accuracies, 's-', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualize sample predictions
model.eval()
with torch.no_grad():
    sample_data = X_test_tensor[:8].to(device)
    sample_targets = y_test_tensor[:8]
    sample_output = model(sample_data)
    _, sample_predicted = torch.max(sample_output, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(8):
    img = sample_data[i].cpu().squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'True: {sample_targets[i]}, Pred: {sample_predicted[i].item()}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()


## Feature Maps Visualization

Let's visualize what the CNN learns by examining feature maps.


In [ ]:
# Visualize learned filters (first convolutional layer)
with torch.no_grad():
    filters = model.conv1.weight.data.cpu().numpy()
    
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i in range(min(32, filters.shape[0])):
    row = i // 8
    col = i % 8
    filter_img = filters[i, 0]  # First channel
    axes[row, col].imshow(filter_img, cmap='gray')
    axes[row, col].axis('off')
    axes[row, col].set_title(f'Filter {i}', fontsize=8)

plt.suptitle('Learned Filters (First Convolutional Layer)', fontsize=14)
plt.tight_layout()
plt.show()

# Visualize feature maps for a sample image
model.eval()
sample_img = X_test_tensor[0:1].to(device)

# Hook to capture feature maps
feature_maps = []
def hook_fn(module, input, output):
    feature_maps.append(output.detach().cpu().numpy())

hook = model.conv1.register_forward_hook(hook_fn)
with torch.no_grad():
    _ = model(sample_img)
hook.remove()

if feature_maps:
    fm = feature_maps[0][0]  # First batch, all channels
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    for i in range(min(32, fm.shape[0])):
        row = i // 8
        col = i % 8
        axes[row, col].imshow(fm[i], cmap='viridis')
        axes[row, col].axis('off')
    
    plt.suptitle('Feature Maps (First Convolutional Layer)', fontsize=14)
    plt.tight_layout()
    plt.show()


## Validation & Testing

Let's validate the model performance.


In [ ]:
# Assertions
assert test_accuracy > 50, "CNN should perform better than random"
assert len(train_losses) == num_epochs, "Should have trained for all epochs"
print("\n✓ Validation checks passed")

# Compare with simple MLP (for reference)
print("\nNote: CNNs are specifically designed for image data.")
print("They use parameter sharing and local connectivity,")
print("making them more efficient than fully connected networks for images.")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Convolutional Layers**
   - Apply filters to detect local patterns
   - Share weights across spatial locations
   - Translation invariant
   - Efficient parameter usage

2. **Pooling Layers**
   - Downsample feature maps
   - Reduce spatial dimensions
   - Max pooling: preserves strongest features
   - Helps with overfitting

3. **CNN Architecture**
   - Stack of conv + activation + pooling
   - Feature extraction layers
   - Classification layers (fully connected)
   - Hierarchical feature learning

4. **Key Components**
   - **Filters/Kernels**: Detect features
   - **Stride**: Step size
   - **Padding**: Border handling
   - **Feature maps**: Output of convolutions

### When to Use CNNs

✅ **Good for:**
- Image classification
- Object detection
- Computer vision tasks
- Data with spatial structure
- Transfer learning
- When you need translation invariance

❌ **Not ideal for:**
- Tabular data (use MLPs)
- Sequential data (use RNNs)
- Very small datasets
- When interpretability is critical
- Real-time applications (can be slow)

### Next Steps

- Explore **Transfer Learning** with pre-trained models
- Try **Data Augmentation** to improve performance
- Experiment with **different architectures** (ResNet, VGG, etc.)
- Apply to **object detection** tasks
- Use **Batch Normalization** for better training
